In [12]:
import pandas as pd
import numpy as np


# ============================================================
# 1. READ DATA
# ============================================================

file_path = "/content/AMS_PVD_final_dataset.xlsx"

df = pd.read_excel(file_path)

# Actual column names in your dataset
P_col = "Peakvalue"
V_col = "Volume"
D_col = "Duration"


# ============================================================
# 2. CONVERT TO NUMERIC
# ============================================================

df[P_col] = pd.to_numeric(df[P_col], errors="coerce")
df[V_col] = pd.to_numeric(df[V_col], errors="coerce")
df[D_col] = pd.to_numeric(df[D_col], errors="coerce")


# Remove rows where P, V or D is missing
df = df.dropna(
    subset=[P_col, V_col, D_col]
).copy()


# ============================================================
# 3. EXTRACT VARIABLES
# ============================================================

P = df[P_col]
V = df[V_col]
D = df[D_col]


# ============================================================
# 4. CALCULATE MEDIAN VALUES
# ============================================================

P_median = P.median()
V_median = V.median()
D_median = D.median()


print("\n============================================")
print("MEDIAN VALUES")
print("============================================")

print(f"Peakvalue median = {P_median}")
print(f"Volume median    = {V_median}")
print(f"Duration median  = {D_median}")


# ============================================================
# 5. HIGH / LOW CLASSIFICATION
#
# HIGH = value >= median
# LOW  = value < median
# ============================================================

df["Peak_Level"] = np.where(
    df[P_col] >= P_median,
    "High",
    "Low"
)

df["Volume_Level"] = np.where(
    df[V_col] >= V_median,
    "High",
    "Low"
)

df["Duration_Level"] = np.where(
    df[D_col] >= D_median,
    "High",
    "Low"
)


# ============================================================
# 6. FLOOD TYPE CLASSIFICATION
#
# P-V:
# High Peak + High Volume + Low Duration
#
# V-D:
# Low Peak + High Volume + High Duration
#
# P-D:
# High Peak + Low Volume + High Duration
#
# EXTREME:
# High Peak + High Volume + High Duration
#
# LOW-INTENSITY:
# Low Peak + Low Volume + Low Duration
# ============================================================

def classify_flood(row):

    P_high = row[P_col] >= P_median
    V_high = row[V_col] >= V_median
    D_high = row[D_col] >= D_median

    # --------------------------------------------------------
    # Extreme event
    # --------------------------------------------------------
    if P_high and V_high and D_high:
        return "Extreme event"

    # --------------------------------------------------------
    # P-V type
    # --------------------------------------------------------
    elif P_high and V_high and not D_high:
        return "P-V type"

    # --------------------------------------------------------
    # V-D type
    # --------------------------------------------------------
    elif not P_high and V_high and D_high:
        return "V-D type"

    # --------------------------------------------------------
    # P-D type
    # --------------------------------------------------------
    elif P_high and not V_high and D_high:
        return "P-D type"

    # --------------------------------------------------------
    # Low-intensity event
    # --------------------------------------------------------
    elif not P_high and not V_high and not D_high:
        return "Low-intensity event"

    # --------------------------------------------------------
    # Remaining combinations
    # --------------------------------------------------------
    else:
        return "Other"


df["Flood_Type"] = df.apply(
    classify_flood,
    axis=1
)


# ============================================================
# 7. SUMMARY
# ============================================================

type_order = [
    "P-V type",
    "V-D type",
    "P-D type",
    "Extreme event",
    "Low-intensity event",
    "Other"
]

summary = (
    df["Flood_Type"]
    .value_counts()
    .reindex(
        type_order,
        fill_value=0
    )
    .reset_index()
)

summary.columns = [
    "Flood Type",
    "Number of Events"
]

summary["Percentage (%)"] = (
    summary["Number of Events"]
    / len(df)
    * 100
)


print("\n============================================")
print("FLOOD CLASSIFICATION SUMMARY")
print("============================================")

print(
    summary.to_string(index=False)
)


# ============================================================
# 8. DOMINANT FLOOD TYPE
# ============================================================

dominant = summary.loc[
    summary["Number of Events"].idxmax()
]

print("\n============================================")
print("DOMINANT FLOOD TYPE")
print("============================================")

print(
    f"Dominant type: {dominant['Flood Type']}"
)

print(
    f"Number of events: "
    f"{int(dominant['Number of Events'])}"
)

print(
    f"Percentage: "
    f"{dominant['Percentage (%)']:.2f}%"
)


# ============================================================
# 9. HIGH / LOW COUNTS
# ============================================================

print("\n============================================")
print("HIGH / LOW COUNTS")
print("============================================")

print(
    f"Peakvalue High: "
    f"{(df[P_col] >= P_median).sum()}"
)

print(
    f"Peakvalue Low : "
    f"{(df[P_col] < P_median).sum()}"
)

print(
    f"Volume High: "
    f"{(df[V_col] >= V_median).sum()}"
)

print(
    f"Volume Low : "
    f"{(df[V_col] < V_median).sum()}"
)

print(
    f"Duration High: "
    f"{(df[D_col] >= D_median).sum()}"
)

print(
    f"Duration Low : "
    f"{(df[D_col] < D_median).sum()}"
)


# ============================================================
# 10. FIRST 10 EVENTS — MANUAL CHECK
# ============================================================

print("\n============================================")
print("FIRST 10 EVENTS — MANUAL CHECK")
print("============================================")

manual_columns = [
    P_col,
    V_col,
    D_col,
    "Peak_Level",
    "Volume_Level",
    "Duration_Level",
    "Flood_Type"
]

print(
    df[manual_columns]
    .head(10)
    .to_string(index=False)
)


# ============================================================
# 11. SAVE CLASSIFIED DATA
# ============================================================

output_columns = [
    P_col,
    V_col,
    D_col,
    "Peak_Level",
    "Volume_Level",
    "Duration_Level",
    "Flood_Type"
]

output_file = "PVD_Median_Flood_Classification.xlsx"

df[output_columns].to_excel(
    output_file,
    index=False
)


print("\n============================================")
print("OUTPUT FILE")
print("============================================")

print(f"Saved as: {output_file}")


MEDIAN VALUES
Peakvalue median = 7213.049999999999
Volume median    = 181040.0
Duration median  = 8.0

FLOOD CLASSIFICATION SUMMARY
         Flood Type  Number of Events  Percentage (%)
           P-V type                 9       19.565217
           V-D type                 2        4.347826
           P-D type                 2        4.347826
      Extreme event                11       23.913043
Low-intensity event                11       23.913043
              Other                11       23.913043

DOMINANT FLOOD TYPE
Dominant type: Extreme event
Number of events: 11
Percentage: 23.91%

HIGH / LOW COUNTS
Peakvalue High: 23
Peakvalue Low : 23
Volume High: 23
Volume Low : 23
Duration High: 24
Duration Low : 22

FIRST 10 EVENTS — MANUAL CHECK
 Peakvalue  Volume  Duration Peak_Level Volume_Level Duration_Level          Flood_Type
    9499.4   95194         3       High          Low            Low               Other
    8953.9  182910        13       High         High           Hig